[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ame5003-nlp/blob/main/labs/lab-01-refresher.ipynb)

**Click the badge above to open this lab in Google Colab.** Then choose *File → Save a copy in Drive* so your work is saved.

# Lab 1 — Refresher and Setup

**MSIS · AME 5053 · Week 1 · 3 hours**

Welcome to the NLP lab.

Today you will not do any NLP. Today you get set up, and you refresh the parts of
Python this course actually leans on.

That sounds dull. It is not. The Python we cover today — strings, dictionaries and
counting — is the Python you will use in every lab after this one. Unit II in
particular is mostly counting, and people who are comfortable with it find the rest
of this course much easier.

**By the end of this lab you will have:**

1. Colab working, with every library this course needs
2. A refresher on strings, dictionaries and counting
3. Seen a working sentiment classifier — and seen it fail

Work through the cells in order. Run every one.

---
## Part 0 — Colab

### 0.1 Where this course runs

Every lab in this course runs in **Google Colab**. You do not install Python. You do not
install anything on your own machine. You open a link and it works.

Colab is a notebook that runs on Google's computers instead of yours. Two reasons we use it:

1. **Everyone gets the identical setup.** The oldest problem in any lab is "it works on my
   machine but not yours". That problem does not exist here.
2. **From week 9 you need a GPU** — a processor built for the maths that neural networks do.
   The lab machines do not have one. Colab lends you one, free.

**What you need:** a Google account and a browser. That is all.

**One thing to understand now, because it surprises everyone:**

> Colab gives you a temporary computer. When your session ends, **that computer is wiped** —
> every library you installed, every file you made. Your *notebook* is saved. Everything
> around it is not.

That sounds bad. It is actually fine, and it is why every notebook in this course starts with
an install cell. You just have to know it.

### 0.2 Install what this course needs

**This cell appears at the top of every lab notebook.** You will run it every time. That is
not a mistake — remember, the machine is wiped between sessions.

It takes about a minute. Run it now.

In [ ]:
# Runs in every lab. The ! sends the line to the shell, not to Python.
# pip skips anything Colab already has, so this is quick.
!pip install -q regex nltk spacy scikit-learn gensim matplotlib
!python -m spacy download en_core_web_sm -q

print()
print("Done. Warnings about dependency conflicts are normal here — ignore them.")

Colab arrives with a lot already installed, so most of that cell will be instant. It installs
only what is missing. You do not need to know which is which — run it and check the next cell.

### 0.3 Check it worked

Nobody leaves the room until every row says `ok`.

In [ ]:
import importlib, sys

REQUIRED = [
    ("regex",      "regular expressions (we use this, NOT the built-in re)"),
    ("nltk",       "tokenizing, stemming, POS — weeks 3 to 5"),
    ("spacy",      "industrial POS and NER — weeks 3, 6, 7"),
    ("sklearn",    "TF-IDF, Naive Bayes — weeks 6, 7"),
    ("gensim",     "Word2Vec and GloVe — week 8"),
    ("torch",      "neural networks — weeks 9 onward"),
    ("numpy",      "arrays, used everywhere"),
    ("matplotlib", "plots — week 8"),
]

try:
    import google.colab
    where = "Google Colab"
except ImportError:
    where = "your own machine (not Colab)"

print(f"Running on: {where}")
print(f"Python:     {sys.version.split()[0]}")
print()
print(f"{'library':<12} {'status':<8} {'version':<12} what it is for")
print("-" * 78)

missing = []
for name, why in REQUIRED:
    try:
        mod = importlib.import_module(name)
        print(f"{name:<12} {'ok':<8} {getattr(mod, '__version__', '?'):<12} {why}")
    except Exception:
        missing.append(name)
        print(f"{name:<12} {'MISSING':<8} {'-':<12} {why}")

print()
if missing:
    print("Not ready. Missing:", ", ".join(missing))
    print("Re-run the install cell above, then run this one again.")
else:
    print("Everything works. You are ready for week 2.")

### 0.4 Save your work, or you will lose it

**Read this bit. Every year somebody loses an afternoon to it.**

Your notebook is not saved automatically the way a Google Doc is. And when your Colab session
ends — you close the tab, or leave it idle too long — the machine is wiped.

Do this now:

1. **File → Save a copy in Drive**

That puts your own copy in your Google Drive, under a folder called `Colab Notebooks`. Work in
*that* copy. It saves as you go, and it is yours.

If you skip this and just type into the shared notebook, your work is gone when the tab
closes.

**Two more things worth knowing:**

- **Files you create are temporary.** If a lab writes `output.txt`, that file dies with the
  session. To keep one, download it (the folder icon on the left) or save it to Drive.
- **Long idle and you get disconnected.** If you leave for a break and come back to a dead
  notebook, do not panic — reconnect, then **Runtime → Run all**. You lose the running state,
  not your code.

---
## Part 1 — Jupyter, and the one thing that will confuse you

A notebook is cells of code you run in any order. That flexibility is the point, and it is
also the trap.

**The notebook remembers everything you have run, not what you can see on screen.**

Run these three cells in order.

In [ ]:
x = 10
print("x is", x)

In [ ]:
x = 99
print("x is now", x)

In [ ]:
# Now scroll UP and re-run the FIRST cell, then run this one.
print("x is", x)

Go and do it. Re-run the first cell, then this last one.

You get `x is 10`, even though the cell above says 99. The notebook does not care about
top-to-bottom order — it cares about the order you *ran* things.

This causes real confusion in labs: your code looks right, the output is wrong, and nothing
is broken except the run order.

**The fix, and get in the habit now:** when something behaves strangely, use
**Runtime → Restart session and run all**. It throws away all memory and runs everything from
the top, exactly as written. If your notebook works after that, it really works.

Every notebook you submit in this course must survive a restart-and-run-all.

⚠️ On Colab, a restart also wipes your installed libraries — so the install cell at the top
runs again. That is exactly why it lives at the top of every notebook.

---
## Part 2 — Strings

All of Unit I is string handling. These are the operations you will actually use.

In [ ]:
s = "  The Old Man the Boat.  "

print(repr(s.strip()))            # remove surrounding whitespace
print(repr(s.strip().lower()))    # case fold
print(s.split())                  # split on any whitespace -> list
print("-".join(["a", "b", "c"]))  # list -> string
print(s.strip()[0:3])             # slicing: characters 0, 1, 2
print(s.strip()[-5:])             # last five characters

In [ ]:
# f-strings: the modern way to build strings
word, count = "cat", 3
print(f"{word!r} appears {count} times")
print(f"{count} squared is {count**2}")

### Exercise 1

Below is a messy line. Clean it up into a list of lowercase words with no punctuation.

Expected output: `['the', 'old', 'man', 'the', 'boat']`

In [ ]:
messy = "  The OLD man,  the BOAT!  "

# YOUR CODE HERE
# Hint: strip, lower, then split. Then deal with the , and !
# Hint: regex.sub is one good way to remove punctuation.

---
## Part 3 — Counting

**This is the most important part of today.**

Unit II looks like several different techniques. It is not. N-grams, Bag of Words and
TF-IDF are all *counting dictionaries* wearing different hats. If counting is comfortable,
Unit II is comfortable.

Start with the long way, so you know what the short way is doing.

In [ ]:
words = ["the", "cat", "sat", "on", "the", "mat", "the", "end"]

# The long way: a dictionary from word -> count
counts = {}
for w in words:
    if w in counts:
        counts[w] += 1
    else:
        counts[w] = 1

print(counts)

In [ ]:
# Slightly shorter, using .get with a default
counts = {}
for w in words:
    counts[w] = counts.get(w, 0) + 1
print(counts)

In [ ]:
# The way you will actually use, for the rest of the course
from collections import Counter

counts = Counter(words)
print(counts)
print()
print("the appears:", counts["the"])
print("zebra appears:", counts["zebra"])   # missing key -> 0, no crash
print("two most common:", counts.most_common(2))
print("total words:", sum(counts.values()))
print("distinct words:", len(counts))

Look at the last two lines closely.

- **total words = 8** — how many word *tokens* there are
- **distinct words = 6** — how many word *types* there are

That distinction has names, and you will be asked about it: **tokens** are instances,
**types** are distinct entries. "the cat sat on the mat the end" is 8 tokens and 6 types.

`Counter` gives you both for free. Remember it.

### Exercise 2

Count the words in the paragraph below and print the five most common.

Then answer, using code: how many tokens? How many types?

In [ ]:
text = """the quick brown fox jumps over the lazy dog
the dog barks and the fox runs away
the quick fox is quick"""

# YOUR CODE HERE
# 1. lowercase it
# 2. split into words (use regex.findall, not re!)
# 3. count them
# 4. print the 5 most common, the token count, and the type count

---
## Part 4 — Comprehensions

You will read a lot of these. They are loops written on one line.

In [ ]:
words = ["the", "cat", "sat", "on", "the", "mat"]

# list comprehension: transform
print([w.upper() for w in words])

# with a filter
print([w for w in words if len(w) == 3])

# transform and filter together
print([w.upper() for w in words if w != "the"])

# dict comprehension
print({w: len(w) for w in set(words)})

In [ ]:
# The same thing, written as a loop. Identical result.
out = []
for w in words:
    if w != "the":
        out.append(w.upper())
print(out)

---
## Part 5 — Files and encodings

You will read text files in every lab. There is one rule.

In [ ]:
# Write a file containing non-English text
with open("sample.txt", "w", encoding="utf-8") as f:
    f.write("The Old Man the Boat.\n")
    f.write("ಕನ್ನಡ ಭಾಷೆ\n")
    f.write("नमस्ते\n")

# Read it back properly
with open("sample.txt", encoding="utf-8") as f:
    for line in f:
        print(repr(line.strip()))

In [ ]:
# Now read the SAME file claiming it is ASCII, and watch it fail
try:
    with open("sample.txt", encoding="ascii") as f:
        print(f.read())
except UnicodeDecodeError as e:
    print("UnicodeDecodeError:", e)
    print()
    print("The bytes are fine. We told Python to read them the wrong way.")

**The rule: always pass `encoding="utf-8"` when you open a text file.**

If you leave it out, Python picks a default based on the machine. Colab is Linux, so it
defaults to UTF-8 and you get away with it. Run the same code on a Windows laptop and the
default may be `cp1252` — and code that worked in the lab throws `UnicodeDecodeError` at home.

Be explicit. It costs fifteen characters and saves an afternoon.

### Where did that file go?

Look at the folder icon in the left sidebar. `sample.txt` is there — Colab gave you a real
filesystem.

But it is a **temporary** one. Close this session, come back tomorrow, and that file is gone.
Same for anything a lab downloads or trains.

To keep a file, either:

- **Download it** — folder icon, then the three dots next to the file, then Download
- **Save it to your Drive** — mount Drive first, then write there:

```python
from google.colab import drive
drive.mount('/content/drive')
# now /content/drive/MyDrive/ is your Google Drive
```

You will not need Drive today. But from week 9 you will train models that take a while, and
losing one to a disconnect hurts. Remember this cell exists.

In [ ]:
import locale, sys
print("Your default text encoding:", locale.getpreferredencoding(False))
print("Filesystem encoding       :", sys.getfilesystemencoding())
print()
print("Whatever it says — pass encoding='utf-8' anyway. Then it does not matter.")

---
## Part 6 — A look ahead

You have done no NLP today. Here is thirty seconds of it, so you know where this is going.

In [ ]:
import nltk
nltk.download("vader_lexicon", quiet=True)

from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def sentiment(sentence):
    score = sia.polarity_scores(sentence)["compound"]
    label = "positive" if score > 0.05 else "negative" if score < -0.05 else "neutral"
    print(f"{score:+.3f}  {label:<8}  {sentence}")

sentiment("This course is going to be brilliant.")
sentiment("This course is going to be terrible.")
sentiment("The food was not bad at all, actually.")

That is a sentiment classifier. It reads a sentence and decides whether it is positive or
negative, and it got all three right — including the last one, where *"not bad"* has to flip
*bad* into a positive.

Impressive. Now watch it fall over.

In [ ]:
sentiment("This course is going to be a nightmare.")

**Neutral.** It thinks *"this course is going to be a nightmare"* is neither positive nor
negative.

You know it is negative. Why doesn't it?

In [ ]:
# This classifier is a hand-written list of words with scores. Let's look inside it.
lexicon = sia.lexicon
print("words in the list:", len(lexicon))
print()
for w in ["brilliant", "terrible", "awful", "love", "hate", "nightmare"]:
    print(f"  {w:<10} -> {lexicon.get(w, 'NOT IN THE LIST')}")

There it is. **"nightmare" is not in the list.**

Someone sat down and wrote out 7,500 words with a score for each. They got "brilliant",
"terrible", "awful", "love" and "hate". They did not think of "nightmare". So the classifier
has no opinion about it, and shrugs.

This is not a bug you can fix by trying harder. It is what rules *are*. Somebody must write
down every word, for every domain, in every language — and they will always miss some. Add
"nightmare" and you will trip over "shambles" tomorrow.

**In week 6 you will build a classifier that learns the words by itself**, from labelled
examples, using Naïve Bayes. Nobody writes a list. Show it enough angry reviews and it works
out that "nightmare" is bad on its own.

That is the difference between rules and machine learning, and it is the story of this
entire course.

---
## Before you go

Check all of these:

- [ ] You saved your own copy: **File → Save a copy in Drive**
- [ ] The check cell in 0.3 prints `ok` on every row
- [ ] You know why we use `regex` and not `re`
- [ ] **Runtime → Restart and run all** works top to bottom with no errors
- [ ] You can explain the difference between a token and a type

**Remember for next week**, because both of these catch people out:

1. **Run the install cell first, every time.** Colab wipes the machine between sessions.
   Nothing you installed today will still be there next week. This is normal.
2. **Work in your own Drive copy**, not the shared notebook.

If any box is unticked, ask now. Next week we start on real text, and everything from here
assumes this works.

**Next lab:** information extraction with regular expressions.